# LA County Greenhouse Gas Emissions Analysis

## Overview

**Data Sources:**
| Source | Coverage | Granularity | Time Range |
|--------|----------|-------------|------------|
| EPA GHGRP (Greenhouse Gas Reporting Program) | Facilities emitting 25k+ metric tons CO2e/yr | Individual facility | 2010-present |
| CARB MRR (Mandatory Reporting Regulation) | California-specific, lower threshold | Facility + sector | 2008-present |
| CARB GHG Inventory | Statewide by sector, scalable to county | Sector | 2000-present |

**Time Windows:** Last year (2025), 5 years (2021-2025), decade (2016-2025), full history (2010-2025)

# Notebook 1: LA County GHG Emissions - Data Acquisition

In [1]:
# ============================================================
# Notebook 1: LA County GHG Emissions - Data Acquisition
# ============================================================

# %% [markdown]
# # LA County Greenhouse Gas Emissions Analysis
# ## Data Acquisition & Cleaning Pipeline

# %% Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import os
from pathlib import Path

# Create project directories
for d in ['data/raw', 'data/processed', 'data/tableau_export']:
   Path(d).mkdir(parents=True, exist_ok=True)

print("Project structure created.")

Project structure created.


In [2]:
# %% Cell 2: Download EPA GHGRP Data
# EPA GHGRP provides facility-level GHG data for large emitters
# API: https://enviro.epa.gov/enviro/efservice/

def fetch_ghgrp_data():
   """
   Fetch EPA GHGRP data for LA County facilities.
   Uses EPA Envirofacts RESTful API.
   """

   # GHGRP data by year - EPA Envirofacts REST API
   # We query the PUB_DIM_FACILITY table for LA County
   base_url = "https://enviro.epa.gov/enviro/efservice"

   # Method 1: Direct CSV download from EPA Flight Tool
   # (More reliable for bulk data)
   flight_url = (
       "https://ghgdata.epa.gov/ghgp/service/export?"
       "q=&state=CA&county=Los%20Angeles"
       "&outputFormat=CSV"
   )

   print("Attempting EPA FLIGHT tool download...")
   try:
       response = requests.get(flight_url, timeout=60)
       if response.status_code == 200:
           df = pd.read_csv(io.StringIO(response.text))
           df.to_csv('data/raw/ghgrp_la_county.csv', index=False)
           print(f"Downloaded {len(df)} facility-year records")
           return df
   except Exception as e:
       print(f"API download failed: {e}")

   # Method 2: Fallback - Download bulk GHGRP data files
   print("Falling back to bulk download...")
   bulk_url = (
       "https://ghgdata.epa.gov/ghgp/service/facilityDownload?"
       "csv&stateId=CA&countyId=Los%20Angeles"
   )

   try:
       response = requests.get(bulk_url, timeout=120)
       df = pd.read_csv(io.StringIO(response.text))
       df.to_csv('data/raw/ghgrp_la_county.csv', index=False)
       print(f"Downloaded {len(df)} records")
       return df
   except Exception as e:
       print(f"Bulk download also failed: {e}")
       print("\n--- MANUAL DOWNLOAD INSTRUCTIONS ---")
       print("1. Go to: https://ghgdata.epa.gov/ghgp/main.do#/facility")
       print("2. Filter: State=California, County=Los Angeles")
       print("3. Click 'Export Data' -> CSV")
       print("4. Save as: data/raw/ghgrp_la_county.csv")
       return None

ghgrp_df = fetch_ghgrp_data()

Attempting EPA FLIGHT tool download...
Falling back to bulk download...
Bulk download also failed: No columns to parse from file

--- MANUAL DOWNLOAD INSTRUCTIONS ---
1. Go to: https://ghgdata.epa.gov/ghgp/main.do#/facility
2. Filter: State=California, County=Los Angeles
3. Click 'Export Data' -> CSV
4. Save as: data/raw/ghgrp_la_county.csv


In [ ]:
# %% Cell 3: Download CARB MRR Data
def fetch_carb_mrr_data():
   """
   Fetch CARB Mandatory Reporting Regulation data.
   Available at: https://ww2.arb.ca.gov/mrr-data
   """

   # CARB MRR data is typically available as Excel files
   # Updated annually
   carb_url = (
       "https://ww2.arb.ca.gov/sites/default/files/classic/cc/reporting/"
       "ghg-rep/reported-data/2022-ghg-emissions-2022-11-04.xlsx"
   )

   print("Downloading CARB MRR data...")
   try:
       response = requests.get(carb_url, timeout=120)
       if response.status_code == 200:
           with open('data/raw/carb_mrr_facilities.xlsx', 'wb') as f:
               f.write(response.content)

           df = pd.read_excel('data/raw/carb_mrr_facilities.xlsx')
           print(f"Downloaded {len(df)} records")
           return df
   except Exception as e:
       print(f"CARB download failed: {e}")
       print("\n--- MANUAL DOWNLOAD INSTRUCTIONS ---")
       print("1. Go to: https://ww2.arb.ca.gov/mrr-data")
       print("2. Download the latest 'Reported Emissions' spreadsheet")
       print("3. Save as: data/raw/carb_mrr_facilities.xlsx")
       return None

carb_df = fetch_carb_mrr_data()

In [ ]:
# %% Cell 4: Download CARB GHG Inventory (Sector-level)
def fetch_carb_inventory():
   """
   CARB's GHG Inventory provides sector-level data for California.
   We'll use this for sector breakdowns and scale to LA County.
   """

   inventory_url = (
       "https://ww2.arb.ca.gov/sites/default/files/classic/cc/inventory/"
       "ghg_inventory_trends_00-20.xlsx"
   )

   print("Downloading CARB GHG Inventory...")
   try:
       response = requests.get(inventory_url, timeout=60)
       if response.status_code == 200:
           with open('data/raw/carb_ghg_inventory.xlsx', 'wb') as f:
               f.write(response.content)
           df = pd.read_excel('data/raw/carb_ghg_inventory.xlsx')
           print(f"Downloaded inventory data")
           return df
   except Exception as e:
       print(f"Download failed: {e}")
       print("\n--- MANUAL DOWNLOAD ---")
       print("1. Go to: https://ww2.arb.ca.gov/ghg-inventory-data")
       print("2. Download 'GHG Inventory - Trends'")
       print("3. Save as: data/raw/carb_ghg_inventory.xlsx")
       return None

inventory_df = fetch_carb_inventory()

In [ ]:
# %% Cell 5: Supplementary - Facility Geocoding Data
def fetch_facility_locations():
   """
   Get lat/lon for LA County facilities for choropleth maps.
   EPA FRS (Facility Registry Service) has this data.
   """

   frs_url = (
       "https://enviro.epa.gov/enviro/efservice/"
       "V_FRS_PROGRAM_FACILITY/STATE_CODE/CA/"
       "COUNTY_NAME/Los%20Angeles/CSV"
   )

   print("Downloading facility location data...")
   try:
       response = requests.get(frs_url, timeout=120)
       if response.status_code == 200:
           df = pd.read_csv(io.StringIO(response.text))
           # Keep relevant columns
           cols_keep = [
               'REGISTRY_ID', 'PRIMARY_NAME', 'LOCATION_ADDRESS',
               'CITY_NAME', 'POSTAL_CODE', 'LATITUDE83', 'LONGITUDE83'
           ]
           available_cols = [c for c in cols_keep if c in df.columns]
           df = df[available_cols].drop_duplicates()
           df.to_csv('data/raw/facility_locations.csv', index=False)
           print(f"Got locations for {len(df)} facilities")
           return df
   except Exception as e:
       print(f"FRS download failed: {e}")

   return None

locations_df = fetch_facility_locations()

In [ ]:
# %% Cell 6: LA County Geographic Boundaries
def fetch_la_boundaries():
   """
   Download LA County zip code / city boundaries for choropleth maps.
   """

   # LA County zip code boundaries (GeoJSON)
   # Source: LA County GIS Data Portal
   geo_url = (
       "https://data.lacounty.gov/api/geospatial/"
       "rnpy-shdb?method=export&type=GeoJSON"
   )

   print("Downloading LA County boundaries...")
   try:
       response = requests.get(geo_url, timeout=60)
       if response.status_code == 200:
           with open('data/raw/la_county_zipcodes.geojson', 'w') as f:
               f.write(response.text)
           print("Saved LA County zip code boundaries")
           return True
   except Exception as e:
       print(f"Boundary download failed: {e}")
       print("\nAlternative: Use Census TIGER/Line files")
       print("https://www.census.gov/cgi-bin/geo/shapefiles/")

   return False

fetch_la_boundaries()

print("\n" + "="*60)
print("DATA ACQUISITION COMPLETE")
print("="*60)

## Notebook 2: Data Processing & Analysis

In [ ]:
# ============================================================
# Notebook 2: Data Processing & Analysis
# ============================================================

# %% Cell 1: Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')